# 🧠 CQ-CNN: GPU-Accelerated Hybrid Classical-Quantum CNN on Kaggle
### Subject-Level (Leakage-Free) Benchmark & 3-Qubit CQ-CNN Training

This notebook trains and evaluates the **Hybrid Classical-Quantum Convolutional Neural Network (CQ-CNN)** for Alzheimer's Disease detection on OASIS-2 MRI volumes using Kaggle's free **T4 GPU**.

**Key Scientific Protocol (Track B)**:
- **Strict Subject-Level Split**: Zero patient overlap between Train, Validation, and Test.
- **Sealed Test Set**: 15 unseen patient brains (never touched during training or scheduling).
- **Parameterized Quantum Variational Circuit**: Qiskit `EstimatorQNN` with `ZZFeatureMap` and `RealAmplitudes` ansatz.
- **Class-Balancing & Calibration**: Addresses the 120:1 natural OASIS-2 class imbalance.

## 1. Environment Setup & Hardware Check

In [ ]:
!nvidia-smi
!pip install --quiet qiskit==2.6.2 qiskit-machine-learning==0.9.1 qiskit-algorithms==0.4.0 nibabel scikit-learn seaborn
print('✓ Dependencies Installed Successfully')

## 2. Clone the Research Repository

In [ ]:
import os
if not os.path.exists('CQ-CNN'):
    !git clone https://github.com/An-Avenger/CQ-CNN.git
os.chdir('CQ-CNN')
print('Current Directory:', os.getcwd())

## 3. Dataset Linking (Kaggle Dataset or GitHub Sample Manifest)

In [ ]:
import pandas as pd
# If using Kaggle input with full preprocessed OASIS-2, point to /kaggle/input/...
# Otherwise, we use the stratified dual-track manifest:
manifest_file = 'data_manifest_sample.csv'
df = pd.read_csv(manifest_file)
print(f'Loaded manifest records: {len(df)}')
print(df.groupby(['track_b_split', 'label']).size())

## 4. Define CQ-CNN Hybrid Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import cat
from qiskit import QuantumCircuit
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit.primitives import StatevectorEstimator as Estimator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Execution Device:', device)

def create_qnn(n_qubits=2):
    feature_map = ZZFeatureMap(n_qubits)
    ansatz = RealAmplitudes(n_qubits, reps=1)
    qc = QuantumCircuit(n_qubits)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)
    return EstimatorQNN(
        circuit=qc,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        input_gradients=True,
        estimator=Estimator()
    )

class CQCNNNet(nn.Module):
    def __init__(self, qnn, n_qubits=2, conv2_ch=4, image_size=128):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 2, kernel_size=5)
        self.conv2 = nn.Conv2d(2, conv2_ch, kernel_size=5)
        self.dropout = nn.Dropout2d(0.25)
        s1 = (image_size - 5 + 1) // 2
        s2 = (s1 - 5 + 1) // 2
        self.fc1 = nn.Linear(conv2_ch * s2 * s2, 4)
        self.fc2 = nn.Linear(4, n_qubits)
        self.qnn = TorchConnector(qnn)
        self.fc3 = nn.Linear(1, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout(x)
        x = x.view(x.shape[0], -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.qnn(x)
        x = self.fc3(x)
        return cat((x, 1 - x), -1)

qnn_2q = create_qnn(n_qubits=2)
model = CQCNNNet(qnn_2q, n_qubits=2, conv2_ch=4).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CQ-CNN Initialized | Trainable Parameters: {total_params:,}')

## 5. Execute Track B Training with Class Weighting

In [ ]:
# Run training script directly with GPU acceleration
!python scirpts/track_b_training.py --arch quantum --qubits 2 --view axial --seeds 1 --manifest data_manifest_sample.csv --weighted --out_dir experiments/kaggle_run

## 6. Evaluate & Plot Test Set Generalization

In [ ]:
import json
summary_path = 'experiments/kaggle_run/quantum_2q_axial/summary.json'
if os.path.exists(summary_path):
    with open(summary_path) as f:
        results = json.load(f)
    print(json.dumps(results, indent=2))
else:
    print('Training summary pending or located at default output directory.')